# Predicción de Fuga de Clientes

El operador de telecomunicaciones **Interconnect** busca reducir la cancelación de clientes (`Churn`) mediante un sistema que identifique con anticipación a quienes podrían darse de baja, para ofrecerles promociones y planes especiales.

**Objetivo:** Desarrollar y comparar modelos capaces de predecir si un cliente se dará de baja próximamente (Sí/No).

El modelo se evaluará principalmente mediante AUC-ROC, utilizando Recall como métrica adicional debido a que resulta importante identificar correctamente a los clientes que abandonan el servicio.

**## Etapa 1: Plan de Trabajo**

**Plan inicial**

El primer paso consiste en explorar y comprender los cuatro conjuntos de datos disponibles y posteriormente integrarlos mediante el identificador único `customer_id`.

Inicialmente se consideró realizar una segmentación de clientes mediante KMeans y posteriormente entrenar modelos de clasificación. Sin embargo, después del análisis exploratorio se priorizó directamente el problema de clasificación supervisada, ya que el objetivo principal del proyecto es predecir la variable Churn.

Se planteó comparar inicialmente:

* Logistic Regression
* LinearSVC
* LightGBM

Posteriormente se incorporaron técnicas para manejar el desbalance de clases y se realizó una búsqueda de hiperparámetros para mejorar el rendimiento.

**1. ¿Cómo unir los datos y qué hacer con los valores nulos?**

Los cuatro archivos se relacionan mediante `customer_id`.

Se utilizó `contract.csv` como tabla principal, ya que contiene los clientes y la variable objetivo. Posteriormente se realizaron uniones `left` con:

* `personal.csv`
* `internet.csv`
* `phone.csv`

Se utilizó `validate='one_to_one'` para comprobar que las relaciones entre las tablas fueran uno a uno.

Después de realizar las uniones se analizaron los valores ausentes generados.

Los valores faltantes de los servicios de Internet se interpretaron como clientes que no cuentan con servicio de Internet, por lo que las variables binarias correspondientes se codificaron como 0 e `internet_service` se completó como `"No internet"`.

De manera similar, los valores faltantes de `multiple_lines` se interpretaron como clientes que no cuentan con este servicio y se codificaron como 0.

Los valores ausentes de `end_date` representan clientes que permanecen activos. Esta variable no se utilizó directamente en el modelo porque contiene información directamente relacionada con la cancelación y podría producir data leakage.

**2. ¿Cuál será la variable objetivo y qué tipo de problema de Machine Learning resolverás?**

La variable objetivo es: `'churn'`

Se transformó en una variable binaria:

0 → cliente activo
1 → cliente que canceló el servicio

Por lo tanto, se trata de un problema de clasificación binaria supervisada.

La proporción de clientes que cancelaron el servicio fue aproximadamente:

Churn = 1 → 26.5 %
Churn = 0 → 73.5 %

Esto representa un desbalance de clases que debe considerarse durante el entrenamiento.

**3. ¿Qué pasos de preprocesamiento e ingeniería de características son necesarios?**

Se realizaron los siguientes pasos:

1. Conversión de nombres de columnas a formato `snake_case`.
2. Conversión de `begin_date` y `end_date` a formato datetime.
3. Transformación de `churn` a variable binaria.
4. Conversión de `total_charges` a variable numérica.
5. Tratamiento de valores ausentes generados durante las uniones.
6. Eliminación de variables que no deben utilizarse directamente para predicción:
    * `customer_id`
    * `begin_date`
    * `end_date`
    * `churn`
7. Codificación de variables categóricas mediante One-Hot Encoding (OHE).
8. Creación de la variable `tenure_months`.

**Ingeniería de características: `tenure_months`**

Se creó una variable que representa aproximadamente la antigüedad del cliente en meses.

Para evitar utilizar información futura, no se calculó la antigüedad utilizando la fecha de cancelación. Se utilizó una fecha de corte común basada en la fecha máxima de inicio registrada en el conjunto de datos.

Se evaluó experimentalmente el impacto de la variable `tenure_months` con y sin ella.

El resultado mostró que esta característica aporta información importante para la predicción:

* Con `tenure_months`: LightGBM alcanzó aproximadamente 0.92–0.93 de ROC-AUC en validación.
* Sin `tenure_months`: el ROC-AUC de LightGBM cayó aproximadamente a 0.82–0.83.

Por esta razón se conservó `tenure_months` en el modelo final.

**4. ¿Qué modelos se entrenaron?**

Se compararon los siguientes modelos:

* Logistic Regression
* LinearSVC
* Random Forest
* LightGBM

Para los modelos se probaron tres estrategias de manejo del desbalance:

* Sin balanceo.
* Ajuste de pesos de clase.
* Oversampling de la clase minoritaria.

LightGBM presentó el mejor desempeño general, por lo que posteriormente se concentró la optimización en este modelo.

**## Etapa 2: Código de Solución**

**### 1. Exploración de Datos (EDA)**

Descripción de los Datos

Los datos están divididos en cuatro archivos:

* `/datasets/final_provider/contract.csv`: Información del contrato (tipo de facturación. método de pago, fechas de inicio y fin).
* `/datasets/final_provider/personal.csv`: Datos demográficos del cliente.
* /datasets/final_provider/internet.csv: Información sobre los servicios de Internet contratados (fibra óptica, DSL. antivirus. etc.).
* `/datasets/final_provider/phone.csv`: Información sobre los servicios telefónicos (líneas múltiples).

Durante el EDA se revisaron:

* Dimensiones de los datasets.
* Tipos de datos.
* Valores ausentes.
* Registros duplicados.
* Distribuciones de variables.
* Valores anómalos.
* Distribución de la variable objetivo.
* Consistencia de las fechas.
* Relación entre las diferentes tablas.

El dataset final quedó compuesto por **7043 clientes.**

La variable objetivo presentó aproximadamente un 26.5 % de clientes con churn, por lo que fue necesario considerar el desbalance durante el entrenamiento.

**2. Preprocesamiento e Ingeniería de Características**

Se realizó la limpieza y transformación de las variables antes del entrenamiento.

Las variables categóricas principales fueron:

* type
* payment_method
* gender
* internet_service

Estas variables se transformaron mediante One-Hot Encoding, utilizando:

OneHotEncoder(handle_unknown='ignore')

Esta técnica permite representar las categorías como variables numéricas binarias y evita problemas cuando una categoría aparece en validación o prueba que no estuvo presente durante el entrenamiento.

Las variables numéricas se conservaron mediante `passthrough`.

También se realizó el tratamiento de `total_charges`, convirtiendo los valores vacíos a valores ausentes y posteriormente a formato numérico.

La variable `end_date` no se utilizó como característica debido a que contiene información directamente relacionada con la cancelación del cliente.

Se creó además `tenure_months` como variable de ingeniería de características.

**3. Selección de Variables y Entrenamiento de Modelos (Baseline)**

Los datos se dividieron en:

75 % → TRAIN

15 % → VALID

10 % → TEST

utilizando `stratify` para conservar aproximadamente la misma proporción de clases en los tres conjuntos.

El conjunto TEST se mantuvo separado para la evaluación final.

Se entrenaron inicialmente los 4 diferentes modelos aplicando 3 técnicas de balanceo deferentes:

* Sin balanceo
* `class_weight` o `scale_pos_weight` (para lgbm)
* oversampling

Los resultados mostraron que LightGBM con oversampling presentó el mejor desempeño en ROC-AUC.

De manera general:

|Modelo	|ROC-AUC aproximado en validación|
|---|---|
|LightGBM|0.924|
|Random Forest| 0.861|
|Logistic Regression| 0.836|
|LinearSVC| 0.833|

LightGBM fue claramente el modelo con mayor capacidad de discriminación entre clientes que cancelan y clientes que permanecen activos.

**4. Optimización y Manejo de Desbalance**

Debido a que solamente aproximadamente el 26.5 % de los clientes pertenecían a la clase churn=1, se probaron diferentes estrategias:

* Sin balanceo.
* Ajuste de pesos de clase.
* Oversampling de la clase minoritaria.

El oversampling produjo el mejor resultado de ROC-AUC para LightGBM durante el proceso de validación.

Posteriormente se optimizaron los hiperparámetros de LightGBM.

La configuración utilizada en el modelo final fue:

```text
LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=75,
    random_state=42,
    verbosity=-1
)
```

El modelo utilizó:

* `tenure_months`
* `One-Hot Encoding`
* Oversampling de la clase minoritaria
* Validación cruzada

Antes de evaluar el conjunto de prueba se realizó una validación cruzada estratificada de 5 folds.

Resultados:

|Fold|ROC-AUC|
|----|----|
|1|0.922171|
|2|0.923144|
|3|0.925368|
|4|0.951454|
|5|0.929531|
|Media|0.930334|
|Desviación estándar|0.012142|

El ROC-AUC medio de **0.930334** mostró que el rendimiento del modelo se mantuvo consistente en diferentes particiones de los datos.

Después de finalizar la selección del modelo, se entrenó el modelo definitivo utilizando TRAIN + VALID y se realizó una única evaluación sobre TEST.

**## Etapa 3: Informe de Solución**

**1. ¿Qué modelo elegiste finalmente y por qué?**

El modelo seleccionado fue LightGBM con oversampling de la clase minoritaria.

Fue seleccionado porque presentó el mejor desempeño de ROC-AUC durante la comparación de modelos y mantuvo un rendimiento consistente durante la validación cruzada.

La configuración final fue:

```
n_estimators = 300
learning_rate = 0.05
num_leaves = 63
max_depth = -1
min_child_samples = 75
```

Además, la variable `tenure_months` demostró aportar información significativa al modelo.

La validación cruzada produjo un ROC-AUC medio de 0.930334 con una desviación estándar de 0.012142

Esto indica un rendimiento elevado y relativamente estable en diferentes particiones del conjunto de entrenamiento.

**2. ¿Cuáles fueron las métricas finales en el conjunto de prueba?**

El modelo final fue evaluado sobre el conjunto TEST, que no había sido utilizado durante la selección de hiperparámetros.

Los resultados fueron:

|Métrica|TEST|
|----|----|
|ROC-AUC|0.941063|
|Recall|0.791444|
|F1|0.800000|
|Precision|0.808743|
|Accuracy|0.895035|

El resultado principal fue:

**ROC-AUC = 0.941063**

Este resultado supera ampliamente el requisito mínimo establecido para el proyecto de AUC-ROC >= 0.75.

Además, el modelo alcanzó un **Recall de 0.791444**, identificando correctamente aproximadamente el 79.1 % de los clientes que efectivamente cancelaron el servicio.

**3. ¿Qué significa el Recall en términos de negocio?**

El Recall obtenido fue = 0.791444 ≈ 79.1 %

Esto significa que, de los clientes que realmente abandonaron Interconnect, el modelo fue capaz de identificar aproximadamente 79 de cada 100 como clientes con riesgo de churn utilizando el umbral de clasificación empleado.

Desde el punto de vista de retención, esto permite utilizar el modelo como una herramienta de priorización para el equipo de marketing.

Por ejemplo, el operador podría utilizar las predicciones para identificar clientes con mayor probabilidad de abandonar y posteriormente:

* Ofrecer promociones personalizadas.
* Proponer cambios de plan.
* Ofrecer descuentos o beneficios temporales.
* Contactar al cliente para conocer posibles problemas con el servicio.
* Priorizar campañas de retención.
* Analizar qué servicios o características están asociados con un mayor riesgo de cancelación.

El modelo no garantiza que un cliente vaya a cancelar, sino que proporciona una estimación de riesgo que puede utilizarse para apoyar las acciones de retención.

Con un Recall cercano al 79 %, todavía existiría aproximadamente un 20.9 % de los clientes que cancelan que el modelo no identificaría bajo este umbral, por lo que el sistema debería utilizarse como herramienta de apoyo y no como único criterio para tomar decisiones comerciales.

**Conclusión**

El proyecto permitió desarrollar un modelo de clasificación para predecir la fuga de clientes de Interconnect.

Después de integrar los cuatro datasets, realizar el análisis exploratorio, tratar los valores ausentes, transformar las variables categóricas, crear la característica tenure_months y comparar diferentes modelos y estrategias de balanceo, LightGBM con oversampling presentó el mejor desempeño.

El modelo final obtuvo:

**ROC-AUC TEST = 0.941063**
**Recall TEST  = 0.791444**

El ROC-AUC obtenido demuestra una alta capacidad del modelo para distinguir entre clientes que abandonan y clientes que permanecen en el servicio. El Recall indica que el modelo consigue detectar aproximadamente el 79 % de los clientes que finalmente cancelan.

Por lo tanto, el modelo puede utilizarse como una herramienta de apoyo para que Interconnect identifique clientes con mayor riesgo de fuga y concentre sus esfuerzos de retención en ellos.

# Carga y eploración

In [11]:
import warnings

import sys
import os
# Le dice python que busque liberrías ahí también
sys.path.append(os.path.join('src'))
import funciones_personales as fp

import pandas as pd
import numpy as np
import time
import re

from sklearn.metrics import (
    f1_score,
    roc_auc_score, 
    classification_report, 
    accuracy_score,
    recall_score,
    precision_score
    )
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

In [12]:
# Importación de datos
df_contract= pd.read_csv('datasets/final_provider/contract.csv')
df_internet= pd.read_csv('datasets/final_provider/internet.csv')
df_personal= pd.read_csv('datasets/final_provider/personal.csv')
df_phone= pd.read_csv('datasets/final_provider/phone.csv')

# Crea listas de los dataframes
lista_dfs= [df_contract, df_internet, df_personal, df_phone]

# Cambia los nombres de columnas de todos los dataframes a snake_case
for i in lista_dfs:
    i.columns = [fp.to_snake_case((col)) for col in i.columns]

In [13]:
# imprime la información general de los df
for i in lista_dfs:
    print(df_personal.info())
    print(df_personal.head(5))
    print(df_personal.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     7043 non-null   object
 1   gender          7043 non-null   object
 2   senior_citizen  7043 non-null   int64 
 3   partner         7043 non-null   object
 4   dependents      7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB
None
  customer_id  gender  senior_citizen partner dependents
0  7590-VHVEG  Female               0     Yes         No
1  5575-GNVDE    Male               0      No         No
2  3668-QPYBK    Male               0      No         No
3  7795-CFOCW    Male               0      No         No
4  9237-HQITU  Female               0      No         No
customer_id       7043
gender               2
senior_citizen       2
partner              2
dependents           2
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entr

## Hallazgos
**df_contract**

* 7,043 datos con diferentes id sin valores nulos

1. Se crea la variable objetivo 'churn' = 1 si el cliente abandonó, churn=0 si sigue siendo cliente usando la columna `'end_date'`.
    * Se aprecia que existe desbalance entre clases: 73% = 0, 27%  = 1. Se tratará después de la concatenación

2. Se transforman las fechas a datetime dejando el string "No" como NaT.
    * OJO estas columnas contienen la respuesta directa por lo que hay que tener cuidado al implementar el modelo.

3. Se transforma `'total_charges` a float y se sustituyen los NaN por 0.0
    * Solo son 11 clientes con un contrato muy reciente los cuales no ha llegado la fecha de facturación aún

**Resto de los df**

* Se comprueba que los customer_id existan en `'df_contracts`´
* Se define una lista de columnas con los datos de "Yes" y "No" transformados a 1 y 0 respectivamente ('gender' no se incluye)

4. Se realiza un merge a través de los `customer_id` y se tratan los valores NaN derivados del merge.

In [14]:
# Se crea la variable objetivo
df_contract['churn']= (df_contract['end_date']!='No').astype(int)

# Transforma columnas begin_date y end_date a datetime
df_contract['begin_date']= pd.to_datetime(
    df_contract['begin_date'],
    format='%Y-%d-%m'
)
df_contract['end_date'] = pd.to_datetime(
    df_contract['end_date'].replace('No', pd.NaT),
    format='%Y-%d-%m %H:%M:%S'
)

# Transforma la columna total changes a float y rellena nan con 0
df_contract['total_charges']= pd.to_numeric(df_contract['total_charges'], errors='coerce').fillna(0.0)

In [15]:
# Revisa balance de clases
print(df_contract['churn'].value_counts())
df_contract['churn'].value_counts(normalize=True) * 100

churn
0    5174
1    1869
Name: count, dtype: int64


churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64

In [16]:
# muestra los clientes con total_charges = 0
fil= df_contract[df_contract['total_charges']==0]
print(fil.info())
print(fil.nunique())
print(fil['begin_date'].unique())
print(fil['churn'].unique())

<class 'pandas.core.frame.DataFrame'>
Index: 11 entries, 488 to 6754
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        11 non-null     object        
 1   begin_date         11 non-null     datetime64[ns]
 2   end_date           0 non-null      datetime64[ns]
 3   type               11 non-null     object        
 4   paperless_billing  11 non-null     object        
 5   payment_method     11 non-null     object        
 6   monthly_charges    11 non-null     float64       
 7   total_charges      11 non-null     float64       
 8   churn              11 non-null     int64         
dtypes: datetime64[ns](2), float64(2), int64(1), object(4)
memory usage: 880.0+ bytes
None
customer_id          11
begin_date            1
end_date              0
type                  2
paperless_billing     2
payment_method        3
monthly_charges      11
total_charges         1
churn      

In [17]:
# Compara los 'customer_id' de todos los df contra df_contracts y muestra si existen faltantes
for nombre, df in {
    'internet': df_internet,
    'personal': df_personal,
    'phone': df_phone
}.items():

    faltantes = df.loc[
        ~df['customer_id'].isin(df_contract['customer_id']),
        'customer_id'
    ]

    print(nombre, len(faltantes))

internet 0
personal 0
phone 0


In [18]:
# Lista de columnas binarias
binary_cols = [
    'paperless_billing',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'partner',
    'dependents',
    'multiple_lines'
]

## 
for df in lista_dfs:
    cols = [col for col in binary_cols if col in df.columns]
    for col in cols:
        df[col] = df[col].map({'Yes': 1, 'No': 0})

print(df_contract.describe())

                          begin_date                       end_date  \
count                           7043                           1869   
mean   2016-11-22 16:33:40.048274688  2019-04-08 09:03:10.690208768   
min              2013-01-10 00:00:00            2019-01-10 00:00:00   
25%              2015-01-06 00:00:00            2019-01-11 00:00:00   
50%              2017-01-09 00:00:00            2019-01-11 00:00:00   
75%              2019-01-04 00:00:00            2019-01-12 00:00:00   
max              2020-01-02 00:00:00            2020-01-01 00:00:00   
std                              NaN                            NaN   

       paperless_billing  monthly_charges  total_charges        churn  
count        7043.000000      7043.000000    7043.000000  7043.000000  
mean            0.592219        64.761692    2279.734304     0.265370  
min             0.000000        18.250000       0.000000     0.000000  
25%             0.000000        35.500000     398.550000     0.000000  


## Resumen estadístico

El conjunto de datos `df_contract` contiene **7,043 clientes** e información sobre sus contratos, facturación, fechas de inicio y fin, cargos mensuales y cargos acumulados.

### Fechas de inicio y fin de los clientes

La variable `begin_date` contiene una fecha para los 7.043 clientes, abarcando el periodo de **2013 a 2020**. La mediana de la fecha de inicio se sitúa alrededor de enero de 2017, lo que indica que el conjunto de datos incluye clientes con duraciones de contrato muy variadas.

La variable `end_date` contiene fechas para **1,869 clientes**, mientras que los clientes restantes no tienen una fecha de fin registrada porque seguían activos en el momento que refleja el conjunto de datos.

El número de clientes con una `end_date` es coherente con la variable objetivo `churn` (abandono), donde:

* **1,869 clientes (26.54 %)** tienen `churn = 1`.
* **5,174 clientes (73.46 %)** tienen `churn = 0`.

Por lo tanto, la variable objetivo presenta un desequilibrio de clases moderado, aspecto que se abordará más adelante durante la fase de desarrollo del modelo.

### Cargos mensuales

`monthly_charges` es una variable numérica que oscila entre **18.25 y 118.75**.

| Estadístico        | Valor |
| ------------------ | ----: |
| Media              | 64.76 |
| Mediana            | 70.35 |
| Q1                 | 35.50 |
| Q3                 | 89.85 |
| Desviación estándar| 30.09 |

La diferencia entre la media y la mediana sugiere que la distribución no es perfectamente simétrica. No obstante, la variable tiene un rango numérico bien definido y se conservará como predictor continuo.

### Cargos totales

`total_charges` oscila entre **0 y 8,684.80**.

| Estadístico        | Valor |
| ------------------ | -------: |
| Media              | 2,279.73 |
| Mediana            | 1,394.55 |
| Q1                 | 398.55 |
| Q3                 | 3,786.60 |
| Desviación estándar | 2,266.79 |

La media es considerablemente superior a la mediana, lo que indica una distribución con sesgo a la derecha. Esto concuerda con el hecho de que los clientes tienen diferentes tiempos de permanencia: aquellos suscritos durante períodos más largos pueden acumular cargos totales sustancialmente mayores.

El conjunto de datos original contenía cadenas vacías en `total_charges`. Estos registros corresponden a clientes recién registrados que aún no habían recibido su primera factura y, por tanto, no habían realizado ningún pago. Estas observaciones no deben interpretarse automáticamente como datos erróneos o como un gasto nulo.

Por consiguiente, se conservará `total_charges` como un predictor potencial. Su relación con la antigüedad del cliente y la tasa de abandono (*churn*) deberá examinarse antes del entrenamiento del modelo.

### Facturación sin papel

`paperless_billing` es una variable binaria codificada como `0/1`.

Su media es **0.5922**, lo que indica que aproximadamente el **59.2 % de los clientes utiliza la facturación sin papel**.

### Variable objetivo

La variable objetivo `churn` es binaria:

* `0`: el cliente no abandonó el servicio.
* `1`: el cliente abandonó el servicio.

La distribución de la variable objetivo es:

| Abandono | Clientes | Porcentaje |
| -------- | -------: | ---------: |
| 0        | 5,174 | 73.46% |
| 1        | 1,869 | 26.54% |

Esta distribución de clases se tendrá en cuenta durante la fase de modelado. El desequilibrio de clases se abordará **una vez preparado el conjunto completo de características y antes del entrenamiento del modelo**, lo que permitirá evaluar los distintos modelos en condiciones comparables.

### Principales hallazgos

El conjunto de datos sobre contratos proporciona varios predictores potencialmente útiles para el abandono, en particular:

* cargos mensuales;
* cargos totales;
* tipo de contrato;
* método de pago;
* facturación sin papel;
* antigüedad del cliente, que puede derivarse de la información de fechas disponible.

No se utilizará `end_date` directamente como predictor, ya que contiene información sobre la salida del cliente y se empleó para definir la propia variable objetivo. Asimismo, `customer_id` se conservará únicamente como identificador para la integración de datos y no se utilizará como característica del modelo. La siguiente etapa consistirá en integrar los conjuntos de datos relativos a contratos, información personal, internet y telefonía; validar los identificadores de los clientes; crear variables derivadas adecuadas, como la antigüedad; codificar las variables categóricas; y preparar la matriz de características final antes de abordar el desequilibrio de clases y comparar los modelos de aprendizaje automático.

In [19]:
# Validación de fechas
print(df_contract[['begin_date', 'end_date']].dtypes)

print('\nBegin date:')
print(df_contract['begin_date'].describe())

print('\nEnd date:')
print(df_contract['end_date'].describe())

# Fechas de finalización anteriores al inicio
fechas_invalidas = df_contract[
    df_contract['end_date'].notna() &
    (df_contract['end_date'] < df_contract['begin_date'])
]

print(f'Fechas inválidas: {len(fechas_invalidas)}')

begin_date    datetime64[ns]
end_date      datetime64[ns]
dtype: object

Begin date:
count                             7043
mean     2016-11-22 16:33:40.048274688
min                2013-01-10 00:00:00
25%                2015-01-06 00:00:00
50%                2017-01-09 00:00:00
75%                2019-01-04 00:00:00
max                2020-01-02 00:00:00
Name: begin_date, dtype: object

End date:
count                             1869
mean     2019-04-08 09:03:10.690208768
min                2019-01-10 00:00:00
25%                2019-01-11 00:00:00
50%                2019-01-11 00:00:00
75%                2019-01-12 00:00:00
max                2020-01-01 00:00:00
Name: end_date, dtype: object
Fechas inválidas: 0


In [20]:
# Concatenación de dfs
df= df_contract.copy()

df= df.merge(
    df_personal,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

df= df.merge(
    df_internet,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

df= df.merge(
    df_phone,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

In [21]:
# Revisión de los Nan posterior al merge
print('Clientes sin internet:', df['internet_service'].isna().sum())
print('Clientes sin teléfono:', df['multiple_lines'].isna().sum())

print(
    'Clientes sin internet en df_internet:',
    (~df['customer_id'].isin(df_internet['customer_id'])).sum()
)

print(
    'Clientes sin teléfono en df_phone:',
    (~df['customer_id'].isin(df_phone['customer_id'])).sum()
)

Clientes sin internet: 1526
Clientes sin teléfono: 682
Clientes sin internet en df_internet: 1526
Clientes sin teléfono en df_phone: 682


Se reemplazarán los nan de las columnas 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv' y 'streaming_movies' con 0

Para internet_service se reemplazará con 'No internet'

In [22]:
# Reemplazo de valores nan
columnas_internet = [
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies'
]

df[columnas_internet] = df[columnas_internet].fillna(0)

df['internet_service'] = df['internet_service'].fillna('No internet')

df['multiple_lines'] = df['multiple_lines'].fillna(0)

# Verificación de valores ausentes en todo el dataset
df.isna().sum().sort_values(ascending=False)

end_date             5174
customer_id             0
begin_date              0
type                    0
paperless_billing       0
payment_method          0
monthly_charges         0
total_charges           0
churn                   0
gender                  0
senior_citizen          0
partner                 0
dependents              0
internet_service        0
online_security         0
online_backup           0
device_protection       0
tech_support            0
streaming_tv            0
streaming_movies        0
multiple_lines          0
dtype: int64

## Añadir datos (Tenure) con justificación
**Importante para añadir tenure sin caer en data leakage**

Si construyesemos tenure asi:
```
df['tenure_months'] = (
    np.where(
        df['end_date'].notna(),
        (df['end_date'] - df['begin_date']).dt.days,
        (fecha_referencia - df['begin_date']).dt.days
    ) / 30.44
)
```
El significado de esta variable sería "Duración total observada de la relación del cliente con la compañía."

Si el objetivo fuera "Clasificar qué clientes terminaron abandonando según las características históricas disponibles en este dataset"

entonces `tenure` calculado con `end_date` puede ser una característica descriptiva.

Pero el objetivo es: **"¿Qué tan probable es que un cliente activo se vaya pronto?"**
Entonces hay un problema. Porque para un cliente que abandonó se estaría calculando su antigüedad hasta después de que ocurrió el evento que se intenta predecir.
```text
Cliente A

2017                 2019
│                     │
Inicio                Abandono
│─────────────────────│
       24 meses
```

El modelo recibe

```
tenure = 24 meses
churn = 1
```

Pero en una predicción real hecha en 2018, todavía no sabríamos que esos 24 meses serían su duración final.

**Diferencia fundamental**

| Variable                   | Qué representa                                | Para "se irá pronto" |
| -------------------------- | --------------------------------------------- | -------------------- |
| `end_date - begin_date`    | Antigüedad **final**                          | ❌ Información futura |
| `fecha_corte - begin_date` | Antigüedad **hasta el momento de predicción** | ✅                    |
| `end_date`                 | Momento en que ocurrió el abandono            | ❌ Leakage directo    |

Por lo que para nuestro objetivo de negocio utilizaremos:

```
fecha_corte = df['begin_date'].max()
```

In [23]:
# Añade tenure_months y verifica que solo haya valores positivos
fecha_corte = df['begin_date'].max()

df['tenure_months'] = (
    (fecha_corte - df['begin_date']).dt.days / 30.44
)

print(df['tenure_months'].describe())

print(
    'Valores negativos:',
    (df['tenure_months'] < 0).sum()
)

print(
    'Valores cero:',
    (df['tenure_months'] == 0).sum()
)

count    7043.000000
mean       37.296648
std        23.653241
min         0.000000
25%        11.925099
50%        35.742444
75%        59.855453
max        83.705650
Name: tenure_months, dtype: float64
Valores negativos: 0
Valores cero: 11


## Procesamiento de datos previo al modelado y a la división

arquitectura propuesta:

```text

                    DATASET COMPLETO
                          │
                          ▼
                  Ingeniería final
                          │
              ┌───────────┴───────────┐
              │                       │
       SIN tenure              CON tenure
              │                       │
              └───────────┬───────────┘
                          │
                          ▼
                  Train / Valid / Test
                     75% / 15% / 10%
                          │
            ┌─────────────┼─────────────┐
            │             │             │
       Sin balanceo   class_weight   Oversampling
            │             │             │
            └─────────────┼─────────────┘
                          │
                          ▼
               Logistic Regression
                    LinearSVC
                    LightGBM
                 Random Forest
                          │
                          ▼
                   comparación
                          │
                          ▼
                 mejor configuración
                          │
                          ▼
                  VALIDACIÓN FINAL
                          │
                          ▼
                       TEST
```

In [24]:
# Crea las variables objetivo y características
# Variables objetivo y características

target = df['churn'].copy()

columnas_excluir = [
    'customer_id',
    'begin_date',
    'end_date',
    'churn'
]

X = df.drop(columns=columnas_excluir).copy()
y = target.copy()

X_sin_tenure = X.drop(
    columns=['tenure_months']
).copy()

print('X con tenure:', X.shape)
print('X sin tenure:', X_sin_tenure.shape)

X con tenure: (7043, 18)
X sin tenure: (7043, 17)


In [25]:
print('Variables categóricas:')
print(X.select_dtypes(include='object').columns.tolist())

print('\nTipos de datos:')
print(X.dtypes)


Variables categóricas:
['type', 'payment_method', 'gender', 'internet_service']

Tipos de datos:
type                  object
paperless_billing      int64
payment_method        object
monthly_charges      float64
total_charges        float64
gender                object
senior_citizen         int64
partner                int64
dependents             int64
internet_service      object
online_security      float64
online_backup        float64
device_protection    float64
tech_support         float64
streaming_tv         float64
streaming_movies     float64
multiple_lines       float64
tenure_months        float64
dtype: object


In [26]:
# split de características

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=54321,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.40,
    random_state=54321,
    stratify=y_temp
)

print('TRAIN:', X_train.shape)
print('VALID:', X_valid.shape)
print('TEST :', X_test.shape)

print('\nChurn TRAIN:')
print(y_train.value_counts(normalize=True))

print('\nChurn VALID:')
print(y_valid.value_counts(normalize=True))

print('\nChurn TEST:')
print(y_test.value_counts(normalize=True))

TRAIN: (5282, 18)
VALID: (1056, 18)
TEST : (705, 18)

Churn TRAIN:
churn
0    0.73457
1    0.26543
Name: proportion, dtype: float64

Churn VALID:
churn
0    0.734848
1    0.265152
Name: proportion, dtype: float64

Churn TEST:
churn
0    0.734752
1    0.265248
Name: proportion, dtype: float64


In [27]:
# Crea las columnas para OHE
## Crea las listas de nombres de columnas categóricas y numéricas
categoricas = [
    'gender',
    'type',
    'payment_method',
    'internet_service'
]

numericas = [
    col for col in X.columns
    if col not in categoricas
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore'
            ),
            categoricas
        ),
        (
            'num',
            'passthrough',
            numericas
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)
X_test_encoded = preprocessor.transform(X_test)

print(type(X_train_encoded))
print(X_train_encoded.shape)

<class 'numpy.ndarray'>
(5282, 26)


Siguiente etapa

```text
                    TRAIN       VALID       TEST

Original             5282        1056        705
                       │
          ┌────────────┼────────────┐
          │            │            │
          ▼            ▼            ▼
      normal       balanced      upsample
```

In [28]:
print('Antes del oversampling:')
print(y_train.value_counts())

X_train_up, y_train_up = fp.upsample_array(
    X_train_encoded,
    y_train.reset_index(drop=True),
    repeat=3
)

print('\nDespués del oversampling:')
print(pd.Series(y_train_up).value_counts())

print('\nProporciones:')
print(pd.Series(y_train_up).value_counts(normalize=True))

print('\nShapes:')
print('Original :', X_train_encoded.shape)
print('Upsampled:', X_train_up.shape)

Antes del oversampling:
churn
0    3880
1    1402
Name: count, dtype: int64

Después del oversampling:
1    4206
0    3880
Name: count, dtype: int64

Proporciones:
1    0.520158
0    0.479842
Name: proportion, dtype: float64

Shapes:
Original : (5282, 26)
Upsampled: (8086, 26)


Hasta ahora contamos con tres condiciones de entrenamiento:

A. X_train_encoded / y_train
   → sin balanceo

B. X_train_encoded / y_train
   → class_weight='balanced'

C. X_train_up / y_train_up
   → oversampling

Comenzaremos con el primer experimento:
Sin hiperparametrización excesiva, para notar diferencias de balanceo con LogisticRegression, LinearSVC y LightGBM como baseline

A continuación se transforman los objetos a ndarray para consistencia (saltarán warnings en el modelo debido a esto)

Para tener:

```text
                     X                     y
Normal       X_train_encoded        y_train_array
Oversampling X_train_up             y_train_up
Validación   X_valid_encoded        y_valid_array
Test         X_test_encoded         y_test_array
```
validación y test no se modifican

In [29]:
y_train_array = y_train.to_numpy()
y_valid_array = y_valid.to_numpy()
y_test_array = y_test.to_numpy()

# Experimento #1, prueba #1: Logistic Regression (Baseline)

In [30]:
# =========================
# 1. Sin balance
# =========================
modelo_lr = LogisticRegression(
    max_iter=2000,
    random_state=42
)

resultado_lr_normal, modelo_lr_normal = fp.evaluar_modelo_clasificacion(
    modelo=modelo_lr,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='Logistic Regression',
    estrategia_balance='Sin balance'
)

# =========================
# 2. Class weight
# =========================
modelo_lr_balanced = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

resultado_lr_balanced, modelo_lr_balanced = fp.evaluar_modelo_clasificacion(
    modelo=modelo_lr_balanced,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='Logistic Regression',
    estrategia_balance='Class weight'
)

# =========================
# 3. oversampling
# =========================
modelo_lr_up = LogisticRegression(
    max_iter=2000,
    random_state=42
)

resultado_lr_up, modelo_lr_up = fp.evaluar_modelo_clasificacion(
    modelo=modelo_lr_up,
    X_train=X_train_up,
    y_train=y_train_up,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='Logistic Regression',
    estrategia_balance='Oversampling'
)

# =========================
# 4. Resultados
# =========================
resultados_lr = pd.concat(
    [
        resultado_lr_normal,
        resultado_lr_balanced,
        resultado_lr_up
    ],
    ignore_index=True
)

resultados_lr.sort_values(
    'F1',
    ascending=False
)

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
1,Logistic Regression,Class weight,Default,0.617605,0.518160,0.764286,0.749053,0.835875,1.504105,0.000324
2,Logistic Regression,Oversampling,Default,0.612994,0.507009,0.775000,0.740530,0.835788,2.153956,0.000433
0,Logistic Regression,Sin balance,Default,0.593750,0.655172,0.542857,0.803030,0.835719,2.416714,0.000357


## Resultados prueba #1 LogReg

1. El balanceo aumenta claramente el Recall.

Sin balance: Recall = 0.543

Con class_weight: Recall = 0.764

Con oversampling: Recall = 0.775

Es decir, al balancear, Logistic Regression detecta muchos más clientes que realmente pertenecen a la clase churn = 1.

2. El precio es una disminución de Precision.

Sin balance: Precision = 0.655

Con class_weight: Precision = 0.518

Con oversampling: Precision = 0.507

Esto significa que el modelo balanceado también está clasificando como potenciales clientes que abandonan a más clientes que finalmente no abandonan.

Es el intercambio típico:

más Recall ↔ menos Precision.

3. F1 mejora con el balanceo.

Sin balance: 0.594
Oversampling: 0.613
Class weight: 0.618

Por tanto, para esta métrica, ``class_weight='balanced'` está funcionando ligeramente mejor que oversampling.

4. ROC-AUC prácticamente no cambia.

Sin balance: 0.835719
Oversampling: 0.835788
Class weight: 0.835875

**Suceso interesante**

El balanceo está cambiando bastante el umbral implícito de clasificación, pero la capacidad de discriminación medida por ROC-AUC permanece prácticamente igual.

En otras palabras: el modelo parece estar ordenando a los clientes de forma muy similar; lo que cambia principalmente es cuántos termina clasificando como churn = 1.

5. Hay una diferencia importante entre Accuracy y F1

Sin balance:

Accuracy = 0.803
F1       = 0.594

`class_weight`:

Accuracy = 0.749
F1       = 0.618

Esta es la razón por la cual no conviene usar Accuracy métrica principal en este problema de churn.

* El modelo sin balance obtiene mayor accuracy porque la clase 0 es mayoritaria. Al intentar detectar mejor la clase minoritaria, el modelo balanceado sacrifica parte de esa accuracy.

**Conclusión provisional**

Para Logistic Regression, hasta el momento:

`class_weight='balanced'` produjo el F1 más alto y un buen equilibrio entre Precision y Recall.

Pero esto no significa todavía que sea nuestro modelo final.

Ahora necesitamos comprobar si este comportamiento se mantiene con modelos no lineales.

## Prueba #2 de balance Linear SVC

In [31]:
# =========================
# 1. Sin balance
# =========================

modelo_svc_normal = LinearSVC(
    random_state=42,
    max_iter=5000
)

resultado_svc_normal, modelo_svc_normal = fp.evaluar_modelo_clasificacion(
    modelo=modelo_svc_normal,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LinearSVC',
    estrategia_balance='Sin balance'
)


# =========================
# 2. Class weight
# =========================

modelo_svc_balanced = LinearSVC(
    class_weight='balanced',
    random_state=42,
    max_iter=5000
)

resultado_svc_balanced, modelo_svc_balanced = fp.evaluar_modelo_clasificacion(
    modelo=modelo_svc_balanced,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LinearSVC',
    estrategia_balance='Class weight'
)


# =========================
# 3. Oversampling
# =========================

modelo_svc_up = LinearSVC(
    random_state=42,
    max_iter=5000
)

resultado_svc_up, modelo_svc_up = fp.evaluar_modelo_clasificacion(
    modelo=modelo_svc_up,
    X_train=X_train_up,
    y_train=y_train_up,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LinearSVC',
    estrategia_balance='Oversampling'
)


# =========================
# Resultados
# =========================

resultados_svc = pd.concat(
    [
        resultado_svc_normal,
        resultado_svc_balanced,
        resultado_svc_up
    ],
    ignore_index=True
)

resultados_svc.sort_values(
    'F1',
    ascending=False
)

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
1,LinearSVC,Class weight,Default,0.612536,0.509479,0.767857,0.742424,0.835130,0.036361,0.000382
2,LinearSVC,Oversampling,Default,0.601671,0.493151,0.771429,0.729167,0.833970,0.041330,0.000458
0,LinearSVC,Sin balance,Default,0.591440,0.649573,0.542857,0.801136,0.833146,0.119909,0.008107


## Reslultados prueba #2 LinearSVC

1. El balanceo vuelve a aumentar Recall

El patrón es prácticamente idéntico:

Sin balance: Recall ≈ 0.543

Class weight: Recall ≈ 0.768

Oversampling: Recall ≈ 0.771

Confirma que no es un comportamiento exclusivo de Logistic Regression.

2. El AUC-ROC se mantiene alrededor de 0.83

Logistic Regression ≈ 0.836
LinearSVC             ≈ 0.835

Esto es una señal bastante buena para continuar con los modelos no lineales.

Pero todavía no podemos decir que el modelo cumple el proyecto, porque falta la evaluación final sobre X_test.

## Prueba #3 LightGBM

La implementación del balanceo cambia ya que LightGBM no cuenta con un metodo `class_weight` per se; en su lugar cuenta con `scale_pos_weight`.

Por lo anterior se utilizará:

sin balance → parámetros normales
balanceado → `scale_pos_weight`
oversampling → `X_train_up`, `y_train_up`

Ya que contamos con diferencias entre clases en `'churn'`(Clase 0 = 3880, Clase 1 = 1402), el peso aproximado de la clase positiva sería: `peso_churn = 3880 / 1402` ~ 2.77.

In [32]:
# obtención de peso "exacto"
peso_churn = (
    (y_train_array == 0).sum()
    / (y_train_array == 1).sum()
)

peso_churn

np.float64(2.767475035663338)

In [33]:
# =========================
# 1. Sin balance
# =========================

modelo_lgbm_normal = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    verbosity=-1
)

resultado_lgbm_normal, modelo_lgbm_normal = fp.evaluar_modelo_clasificacion(
    modelo=modelo_lgbm_normal,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LightGBM',
    estrategia_balance='Sin balance'
)

resultado_lgbm_normal

# =========================
# 2. scale_pos_weight
# =========================

modelo_lgbm_balanced = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    scale_pos_weight=peso_churn,
    random_state=42,
    verbosity=-1
)

resultado_lgbm_balanced, modelo_lgbm_balanced = fp.evaluar_modelo_clasificacion(
    modelo=modelo_lgbm_balanced,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LightGBM',
    estrategia_balance='Class weight'
)

resultado_lgbm_balanced

# =========================
# 3. Oversampling
# =========================

modelo_lgbm_up = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    verbosity=-1
)

resultado_lgbm_up, modelo_lgbm_up = fp.evaluar_modelo_clasificacion(
    modelo=modelo_lgbm_up,
    X_train=X_train_up,
    y_train=y_train_up,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LightGBM',
    estrategia_balance='Oversampling'
)

resultado_lgbm_up

# =========================
# Resultados
# =========================

resultados_lgbm = pd.concat(
    [
        resultado_lgbm_normal,
        resultado_lgbm_balanced,
        resultado_lgbm_up
    ],
    ignore_index=True
)

resultados_lgbm.sort_values(
    'F1',
    ascending=False
)

/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/ut

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
1,LightGBM,Class weight,Default,0.759931,0.735786,0.785714,0.868371,0.920098,0.852680,0.018771
0,LightGBM,Sin balance,Default,0.758197,0.889423,0.660714,0.888258,0.923946,0.532898,0.018803
2,LightGBM,Oversampling,Default,0.754653,0.717042,0.796429,0.862689,0.927407,0.661523,0.020847


In [34]:
resultados_base = pd.concat(
    [
        resultados_lr,
        resultados_svc,
        resultados_lgbm
    ],
    ignore_index=True
)

resultados_base.sort_values(
    'ROC-AUC',
    ascending=False
)

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
8,LightGBM,Oversampling,Default,0.754653,0.717042,0.796429,0.862689,0.927407,0.661523,0.020847
6,LightGBM,Sin balance,Default,0.758197,0.889423,0.660714,0.888258,0.923946,0.532898,0.018803
7,LightGBM,Class weight,Default,0.759931,0.735786,0.785714,0.868371,0.920098,0.852680,0.018771
1,Logistic Regression,Class weight,Default,0.617605,0.518160,0.764286,0.749053,0.835875,1.504105,0.000324
2,Logistic Regression,Oversampling,Default,0.612994,0.507009,0.775000,0.740530,0.835788,2.153956,0.000433
0,Logistic Regression,Sin balance,Default,0.593750,0.655172,0.542857,0.803030,0.835719,2.416714,0.000357
4,LinearSVC,Class weight,Default,0.612536,0.509479,0.767857,0.742424,0.835130,0.036361,0.000382
5,LinearSVC,Oversampling,Default,0.601671,0.493151,0.771429,0.729167,0.833970,0.041330,0.000458
3,LinearSVC,Sin balance,Default,0.591440,0.649573,0.542857,0.801136,0.833146,0.119909,0.008107


## Resultados prueba #3

**Hágase la luz dijo LightGBM (mejor candidato hasta el momento)**

| Estrategia   |        F1 | Precision |    Recall |  Accuracy |   ROC-AUC |
| ------------ | --------: | --------: | --------: | --------: | --------: |
| Oversampling |     0.755 |     0.717 | **0.796** |     0.863 | **0.927** |
| Sin balance  | **0.758** | **0.889** |     0.661 | **0.888** |     0.924 |
| Class weight | **0.760** |     0.736 |     0.786 |     0.868 |     0.920 |

**¿Qué nos dice esto?**

1. LightGBM está muy por encima de los modelos lineales, respecto a ROC-AUC:

Logistic Regression    ≈ 0.836
LinearSVC              ≈ 0.835
LightGBM               ≈ 0.920 – 0.927

* Los tres experimentos de LightGBM superan 0.88 en validación, con lo que ya estaría obteniendo un sobresaliente, pero... ...todavía no quiero declarar el proyecto como sobresaliente, porque el criterio oficial se evalúa sobre **TEST**, no sobre **VALID**.

2. **Algo interesante**: El balanceo ya no tiene el mismo efecto que en los dos anteriores

Con Logistic Regression y LinearSVC:

balancear → aumenta Recall → disminuye Precision.

Mismo comportamiento en LightGBM, pero, con una diferencia importante:

Sin balance:
**Precision = 0.889**
Recall    = 0.661
F1        = 0.758

Es muy conservador al marcar clientes como churn.

Oversampling:
Precision = 0.717
**Recall    = 0.796**
F1        = 0.755

Detecta muchos más churn, pero comete más falsos positivos.

Class weight (scale_pos_weight):
Precision = 0.736
Recall    = 0.786
F1        = 0.760

Consigue un equilibrio bastante bueno entre ambos.

3. Cuestión importante con ROC-AUC

Oversampling: 0.927407
Sin balance: 0.923946
Class weight: 0.920098

Mientras que F1:

Class weight: 0.759931
Sin balance: 0.758197
Oversampling: 0.754653

Por lo anterior, se infiere que la estrategia que maximiza F1 no es la misma que maximiza ROC-AUC.

**ROC-AUC**: Evalúa la capacidad del modelo para ordenar/discriminar entre clientes que abandonan y los que no, considerando distintos umbrales.

**F1**: Evalúa el equilibrio entre Precision y Recall para el umbral de clasificación utilizado.

Entonces, es erroneo pensar "Class weight ganó porque tiene el F1 más alto.". Y a que para esteproyecto, se necesita considerar explícitamente el requisito de AUC-ROC en test.

4. El candidato más interesante

Por ahora, la configuración que merece una atención especial es:

**LightGBM** Oversampling por sus resultados en ROC-AUC = 0.9274 y F1 = 0.7547; sin embbargo, todavía no se debe seleccionar como modelo final ya que aún hace falta implementar el modelo Random Forest (sólo con variación en balance), comparar tenure_months vs. sin tenure_months (respondiendo a la pregunta si esta variable agregada ayuda o "ensucia" al modelo) y ajustar hiperparámetros. De esta manera poder validar de manera más robusta las configuraciones candidatas. Terminando con la elección de un modelo con configuración final y evaluarla una sola vez en TEST.

## Prueba #4 Random Forest

In [35]:
# =========================
# 1. Sin balance
# =========================

modelo_rf_normal = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=54321,
    n_jobs=-1
)

resultado_rf_normal, modelo_rf_normal = fp.evaluar_modelo_clasificacion(
    modelo=modelo_rf_normal,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='Random Forest',
    estrategia_balance='Sin balance'
)

resultado_rf_normal

# =========================
# 2. Class weight
# =========================

modelo_rf_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=54321,
    n_jobs=-1
)

resultado_rf_balanced, modelo_rf_balanced = fp.evaluar_modelo_clasificacion(
    modelo=modelo_rf_balanced,
    X_train=X_train_encoded,
    y_train=y_train_array,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='Random Forest',
    estrategia_balance='Class weight'
)

resultado_rf_balanced

# =========================
# 3. Oversampling
# =========================

modelo_rf_up = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=54321,
    n_jobs=-1
)

resultado_rf_up, modelo_rf_up = fp.evaluar_modelo_clasificacion(
    modelo=modelo_rf_up,
    X_train=X_train_up,
    y_train=y_train_up,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='Random Forest',
    estrategia_balance='Oversampling'
)

resultado_rf_up

# =========================
# Resultados
# =========================

resultados_rf = pd.concat(
    [
        resultado_rf_normal,
        resultado_rf_balanced,
        resultado_rf_up
    ],
    ignore_index=True
)

resultados_rf.sort_values(
    'ROC-AUC',
    ascending=False
)

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
0,Random Forest,Sin balance,Default,0.640927,0.697479,0.592857,0.823864,0.860967,3.679484,0.367550
2,Random Forest,Oversampling,Default,0.647328,0.565333,0.757143,0.781250,0.860567,3.809738,0.399355
1,Random Forest,Class weight,Default,0.649600,0.588406,0.725000,0.792614,0.858091,3.671480,0.408987


## Tabla experimento base completa

In [36]:
resultados_modelos = pd.concat(
    [
        resultados_lr,
        resultados_svc,
        resultados_lgbm,
        resultados_rf
    ],
    ignore_index=True
)

resultados_modelos = resultados_modelos.sort_values(
    'ROC-AUC',
    ascending=False
).reset_index(drop=True)

resultados_modelos

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
0,LightGBM,Oversampling,Default,0.754653,0.717042,0.796429,0.862689,0.927407,0.661523,0.020847
1,LightGBM,Sin balance,Default,0.758197,0.889423,0.660714,0.888258,0.923946,0.532898,0.018803
2,LightGBM,Class weight,Default,0.759931,0.735786,0.785714,0.868371,0.920098,0.852680,0.018771
3,Random Forest,Sin balance,Default,0.640927,0.697479,0.592857,0.823864,0.860967,3.679484,0.367550
4,Random Forest,Oversampling,Default,0.647328,0.565333,0.757143,0.781250,0.860567,3.809738,0.399355
5,Random Forest,Class weight,Default,0.649600,0.588406,0.725000,0.792614,0.858091,3.671480,0.408987
6,Logistic Regression,Class weight,Default,0.617605,0.518160,0.764286,0.749053,0.835875,1.504105,0.000324
7,Logistic Regression,Oversampling,Default,0.612994,0.507009,0.775000,0.740530,0.835788,2.153956,0.000433
8,Logistic Regression,Sin balance,Default,0.593750,0.655172,0.542857,0.803030,0.835719,2.416714,0.000357
9,LinearSVC,Class weight,Default,0.612536,0.509479,0.767857,0.742424,0.835130,0.036361,0.000382


## Conclusiones Experimento 1

tres grupos bien definidos:
|Modelo|ROC-AUC|
|------|-------|
|LightGBM|0.920 - 0.927|
|Random Forest|0.858 - 0.861|
|Modelos lineales|0.833 - 0.836|

Esto sugiere que las variables del dataset cuentan con componentes no lineales respecto a `'churn'`.

Para la siguiente etapa se descartarán solo los modelos lineales, ya que Random Forest muestra potencial.

El siguiente experimento consiste en analizar si la variable agregada `tenure_months` ayuda o "ensucia" el modelo; para ello entrenaremos LightGMB y RandomForest sin esta variable y compararemos resultados.


# Experimento #2 `tenure_months`

Para el entrenamiento sin_tenure se utilizarán:

```
y_train_array = y_train.to_numpy()
y_valid_array = y_valid.to_numpy()
y_test_array = y_test.to_numpy()
```

definidos previamente, y para el `'scale_pos_weight'` el `'peso_churn'` también definido anteriormente.

```text
                    ┌── LightGBM + tenure
                    │       ├── sin balance
                    │       ├── class weight
                    │       └── oversampling
Datos ──────────────┤
                    └── LightGBM - tenure
                            ├── sin balance
                            ├── class weight
                            └── oversampling
                                      │
                                      ▼
                              Comparar ROC-AUC
                                      │
                                      ▼
                              Mejor configuración
                                      │
                         ┌────────────┴────────────┐
                         ▼                         ▼
                  Random Forest              LightGBM
                  comprobación               tuning
                         │                         │
                         └────────────┬────────────┘
                                      ▼
                              Selección final
                                      │
                                      ▼
                                   TEST
```

In [37]:
# Crear datos sin `'tenure_months'`
X_train_sin_tenure = X_train.drop(columns=['tenure_months']).copy()

X_valid_sin_tenure = X_valid.drop(columns=['tenure_months']).copy()

X_test_sin_tenure = X_test.drop(columns=['tenure_months']).copy()

# Comprobación de cambios
print('TRAIN:', X_train_sin_tenure.shape)
print('VALID:', X_valid_sin_tenure.shape)
print('TEST :', X_test_sin_tenure.shape)

# OHE para los nuevos datos
categoricas_sin_tenure = [
    'gender',
    'type',
    'payment_method',
    'internet_service'
]

numericas_sin_tenure = [
    col
    for col in X_train_sin_tenure.columns
    if col not in categoricas_sin_tenure
]

preprocessor_sin_tenure = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categoricas_sin_tenure
        ),
        (
            'num',
            'passthrough',
            numericas_sin_tenure
        )
    ]
)

# Transformación de datos con fit solo en train
X_train_encoded_sin_tenure = (
    preprocessor_sin_tenure.fit_transform(
        X_train_sin_tenure
    )
)

X_valid_encoded_sin_tenure = (
    preprocessor_sin_tenure.transform(
        X_valid_sin_tenure
    )
)

X_test_encoded_sin_tenure = (
    preprocessor_sin_tenure.transform(
        X_test_sin_tenure
    )
)

# Comprobación de transformaciones
print('TRAIN:', X_train_encoded_sin_tenure.shape)
print('VALID:', X_valid_encoded_sin_tenure.shape)
print('TEST :', X_test_encoded_sin_tenure.shape)

TRAIN: (5282, 17)
VALID: (1056, 17)
TEST : (705, 17)
TRAIN: (5282, 25)
VALID: (1056, 25)
TEST : (705, 25)


In [38]:
# Realizar el oversampling al conjunto sin tenure
X_train_up_sin_tenure, y_train_up_sin_tenure = fp.upsample_array(
    X_train_encoded_sin_tenure,
    y_train_array,
    repeat=3
)

# Comprobación de cambios y balance
print('Antes:')
print(pd.Series(y_train_array).value_counts())

print('\nDespués de oversampling:')
print(pd.Series(y_train_up_sin_tenure).value_counts())

print('\nShape:')
print(X_train_up_sin_tenure.shape)

Antes:
0    3880
1    1402
Name: count, dtype: int64

Después de oversampling:
1    4206
0    3880
Name: count, dtype: int64

Shape:
(8086, 25)


In [39]:
# Entrenamiento de los modelos
# =========================
# 1. Sin balance
# =========================
modelo_lgbm_sin_tenure = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    verbosity=-1
)

resultado_lgbm_sin_tenure, modelo_lgbm_sin_tenure = (
    fp.evaluar_modelo_clasificacion(
        modelo=modelo_lgbm_sin_tenure,
        X_train=X_train_encoded_sin_tenure,
        y_train=y_train_array,
        X_valid=X_valid_encoded_sin_tenure,
        y_valid=y_valid_array,
        nombre_modelo='LightGBM',
        estrategia_balance='Sin balance'
    )
)

# =========================
# 2. scale_pos_weight
# =========================
modelo_lgbm_weight_sin_tenure = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    scale_pos_weight=peso_churn,
    random_state=42,
    verbosity=-1
)

resultado_lgbm_weight_sin_tenure, modelo_lgbm_weight_sin_tenure = (
    fp.evaluar_modelo_clasificacion(
        modelo=modelo_lgbm_weight_sin_tenure,
        X_train=X_train_encoded_sin_tenure,
        y_train=y_train_array,
        X_valid=X_valid_encoded_sin_tenure,
        y_valid=y_valid_array,
        nombre_modelo='LightGBM',
        estrategia_balance='Class weight'
    )
)

# =========================
# 3. Oversampling
# =========================
modelo_lgbm_up_sin_tenure = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    verbosity=-1
)

resultado_lgbm_up_sin_tenure, modelo_lgbm_up_sin_tenure = (
    fp.evaluar_modelo_clasificacion(
        modelo=modelo_lgbm_up_sin_tenure,
        X_train=X_train_up_sin_tenure,
        y_train=y_train_up_sin_tenure,
        X_valid=X_valid_encoded_sin_tenure,
        y_valid=y_valid_array,
        nombre_modelo='LightGBM',
        estrategia_balance='Oversampling'
    )
)

# =========================
# 3. Resultados
# =========================

resultados_lgbm_sin_tenure = pd.concat(
    [
        resultado_lgbm_sin_tenure,
        resultado_lgbm_weight_sin_tenure,
        resultado_lgbm_up_sin_tenure
    ],
    ignore_index=True
)

resultados_lgbm_sin_tenure.sort_values(
    'ROC-AUC',
    ascending=False
).reset_index(drop=True)

/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/ut

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
0,LightGBM,Oversampling,Default,0.615616,0.531088,0.732143,0.757576,0.825605,0.977505,0.050680
1,LightGBM,Sin balance,Default,0.558704,0.644860,0.492857,0.793561,0.825573,0.540506,0.027947
2,LightGBM,Class weight,Default,0.606707,0.529255,0.710714,0.755682,0.822218,0.786808,0.054251


## Conclusión Experimeto #2 Tenure

La diferencia es muy grande:

Con oversampling sin tenure ROC-AUC cae 0.9274 → 0.8256. Aproximadamente 0.102 puntos de AUC.

Existe evidencia fuerte de que `'tenure_months'` está aportando información predictiva importante. Sin embargo, no significa que se haya demostrado que tenure sea "la causa" de la mejora. Lo que demuestra experimentalmente es que: Dentro de este conjunto de datos y bajo este esquema de entrenamiento, eliminar `'tenure_months'` reduce considerablemente el desempeño de LightGBM.

Se toma la decisión de conservar `'tenure_months'` para el resto del proyecto.

Ahora sí: volvemos a la pregunta de Random Forest.

## Siguientes pasos (Experimento 3)

1. Optimizar LightGBM con `'tenure_months'`.

* Antes de realizar un GridSearch gigante, se propone una búsqueda controlada sobre:

```
n_estimators
learning_rate
num_leaves
max_depth
min_child_samples
subsample
colsample_bytree
reg_alpha
reg_lambda
```
* No se busca optimizar solamente F1 ya que el crtiterio principal del proyecto es ROC-AUC, sin embargo se seguirán registrando para ayudar a la interpretación de los modelos.

2. Random Forest se dejará como modelo de referencia y el enfoque se centrará en pulir LighGBM.


# Experimento #3 Pulimiento de hiperparámetros

1. Se explorarán los parámetros que controlan la estructura y capacidad del árbol. ( `n_estimators`, `learning_rate`, `num_leaves`, `max_depth`, `min_child_samples`)

2. De encontrarse una región prometedora, se refinará el modelo con `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`.

Durante la búsqueda se mantendrán `random_state= 42` y `verbosity= -1`

## Búsqueda #1

In [40]:
# Diccionario de hiperparámetros
parametros_lgbm_1 = [
    {
        'n_estimators': 100,
        'learning_rate': 0.05,
        'num_leaves': 15,
        'max_depth': -1,
        'min_child_samples': 20
    },
    {
        'n_estimators': 200,
        'learning_rate': 0.05,
        'num_leaves': 15,
        'max_depth': -1,
        'min_child_samples': 20
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 15,
        'max_depth': -1,
        'min_child_samples': 20
    },

    {
        'n_estimators': 100,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': -1,
        'min_child_samples': 20
    },
    {
        'n_estimators': 200,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': -1,
        'min_child_samples': 20
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': -1,
        'min_child_samples': 20
    },

    {
        'n_estimators': 100,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 20
    },
    {
        'n_estimators': 200,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 20
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 20
    }
]

In [41]:
# Primera búsqueda
resultados_lgbm_1, mejor_lgbm_1 = fp.probar_hiperparametros_por_auc(
    modelo_base=LGBMClassifier,
    lista_parametros=parametros_lgbm_1,
    X_train=X_train_up,
    y_train=y_train_up,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LightGBM',
    estrategia_balance='Oversampling'
)

resultados_lgbm_1

/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/ut

,Modelo,Balance,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s),Tiempo predicción (s)
0,LightGBM,Oversampling,"n_estimators=300, learning_rate=0.05, num_leav...",0.778761,0.771930,0.785714,0.881629,0.931383,0.839594,0.044550
1,LightGBM,Oversampling,"n_estimators=300, learning_rate=0.05, num_leav...",0.783427,0.828685,0.742857,0.891098,0.929639,1.933012,0.059364
2,LightGBM,Oversampling,"n_estimators=200, learning_rate=0.05, num_leav...",0.754653,0.717042,0.796429,0.862689,0.927407,0.613065,0.020576
3,LightGBM,Oversampling,"n_estimators=200, learning_rate=0.05, num_leav...",0.774312,0.796226,0.753571,0.883523,0.926367,1.058327,0.046893
4,LightGBM,Oversampling,"n_estimators=300, learning_rate=0.05, num_leav...",0.749591,0.691843,0.817857,0.855114,0.918709,1.362823,0.078877
5,LightGBM,Oversampling,"n_estimators=100, learning_rate=0.05, num_leav...",0.731624,0.701639,0.764286,0.851326,0.914820,0.516755,0.013831
6,LightGBM,Oversampling,"n_estimators=100, learning_rate=0.05, num_leav...",0.714286,0.642857,0.803571,0.829545,0.913770,0.522670,0.016947
7,LightGBM,Oversampling,"n_estimators=200, learning_rate=0.05, num_leav...",0.708268,0.628809,0.810714,0.822917,0.911036,0.628635,0.022022
8,LightGBM,Oversampling,"n_estimators=100, learning_rate=0.05, num_leav...",0.678625,0.583548,0.810714,0.796402,0.899429,0.442351,0.016376


## Resultados búsqueda #1

**Record actual en ROC-AUC: 0.9313**

* Profundidad óptima 300 árboles
* `num_leaves=15` es demasiado restrictivo para el modelo
* `num_leaves=31` y `num_leaves=63` ambos funcionan bien
    * El ROC-AUC más alto se da en `num_leaves=63`
    * F1 y precisión son mayores en `num_leaves=31`

Ya que el proyecto se enfoca en el ROC-AUC, se elige como parámetros fijos para la segunda búsqueda a 300 árboles y 63 leaves.

Se probará variando `min_child_samples` dejando fijos `n_estimators = 300`, `learning_rate = 0.05` y `num_leaves = 63`

## Búsqueda #2

In [42]:
# Segunda búsqueda
parametros_lgbm_2 = [
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 10
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 20
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 30
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 50
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 75
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 100
    }
]

# Entrenamientos
resultados_lgbm_2, mejor_lgbm_2 = fp.probar_hiperparametros_por_auc(
    modelo_base=LGBMClassifier,
    lista_parametros=parametros_lgbm_2,
    X_train=X_train_up,
    y_train=y_train_up,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LightGBM',
    estrategia_balance='Oversampling'
)

# Tabla resultados
resultados_lgbm_2[
    [
        'Parámetros',
        'F1',
        'Precision',
        'Recall',
        'Accuracy',
        'ROC-AUC',
        'Tiempo entrenamiento (s)'
    ]
].sort_values(
    'ROC-AUC',
    ascending=False
).reset_index(drop=True)

/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/ut

,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s)
0,"n_estimators=300, learning_rate=0.05, num_leav...",0.790441,0.814394,0.767857,0.892045,0.935544,1.239439
1,"n_estimators=300, learning_rate=0.05, num_leav...",0.780220,0.800752,0.760714,0.886364,0.933528,1.196453
2,"n_estimators=300, learning_rate=0.05, num_leav...",0.785978,0.812977,0.760714,0.890152,0.932534,1.871250
3,"n_estimators=300, learning_rate=0.05, num_leav...",0.799257,0.833333,0.767857,0.897727,0.930725,2.267639
4,"n_estimators=300, learning_rate=0.05, num_leav...",0.779661,0.824701,0.739286,0.889205,0.929699,2.855905
5,"n_estimators=300, learning_rate=0.05, num_leav...",0.783427,0.828685,0.742857,0.891098,0.929639,2.313235


## Resultados búsqueda #2

**Record actual de ROC-AUC: 0.9335**

* se da con `min_child_samples=75`

Para la tercera búsqueda se fijara todo y variará el número de `min_child_samples` alrededor de 75

Posteriormente y previo al test se realiará una validación cruzada estratificada para revisar sobreajuste

## Búsqueda #3

In [43]:
# Búsqueda 3
parametros_lgbm_3 = [
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 40
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 50
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 60
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 70
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 75
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 80
    },
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': -1,
        'min_child_samples': 90
    }
]

# Entrenamientos
resultados_lgbm_3, mejor_lgbm_3 = fp.probar_hiperparametros_por_auc(
    modelo_base=LGBMClassifier,
    lista_parametros=parametros_lgbm_3,
    X_train=X_train_up,
    y_train=y_train_up,
    X_valid=X_valid_encoded,
    y_valid=y_valid_array,
    nombre_modelo='LightGBM',
    estrategia_balance='Oversampling'
)

# Tabla resultados
resultados_lgbm_3[
    [
        'Parámetros',
        'F1',
        'Precision',
        'Recall',
        'Accuracy',
        'ROC-AUC',
        'Tiempo entrenamiento (s)'
    ]
].sort_values(
    'ROC-AUC',
    ascending=False
).reset_index(drop=True)

/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/alienfibio_linux/miniconda3/envs/m_learning/lib/python3.9/site-packages/sklearn/ut

,Parámetros,F1,Precision,Recall,Accuracy,ROC-AUC,Tiempo entrenamiento (s)
0,"n_estimators=300, learning_rate=0.05, num_leav...",0.790441,0.814394,0.767857,0.892045,0.935544,1.548609
1,"n_estimators=300, learning_rate=0.05, num_leav...",0.794063,0.826255,0.764286,0.894886,0.935070,1.455133
2,"n_estimators=300, learning_rate=0.05, num_leav...",0.786765,0.810606,0.764286,0.890152,0.934867,2.350510
3,"n_estimators=300, learning_rate=0.05, num_leav...",0.787546,0.808271,0.767857,0.890152,0.933708,1.464552
4,"n_estimators=300, learning_rate=0.05, num_leav...",0.785978,0.812977,0.760714,0.890152,0.932534,1.462047
5,"n_estimators=300, learning_rate=0.05, num_leav...",0.776340,0.804598,0.750000,0.885417,0.931683,1.388002
6,"n_estimators=300, learning_rate=0.05, num_leav...",0.777570,0.815686,0.742857,0.887311,0.930978,1.711462


## Resultados búsqueda #3

El mejor modelo de la búsqueda 2 coincide con el de búsqueda 3 por lo que este será el utilizado para una validación cruzada previa al test para así comprobar la estabilidad del modelo.

El oversampling se realizará dentro de cada fold para evitar duplicados

```text
Fold
 │
 ├── Fold-train
 │      ↓
 │   Oversampling
 │      ↓
 │   Modelo
 │
 └── Fold-validation
        ↓
      evaluación
```
Para la CV se usarán 

```
X_train_cv = X_train.copy()
y_train_cv = y_train.copy()
```



In [44]:
# datos
X_train_cv = X_train.copy()
y_train_cv = y_train.copy()

# fold
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=54321
)

resultados_cv = []

for fold, (idx_train, idx_valid) in enumerate(
    skf.split(X_train_cv, y_train_cv),
    start=1
):

    X_fold_train = X_train_cv.iloc[idx_train].copy()
    X_fold_valid = X_train_cv.iloc[idx_valid].copy()

    y_fold_train = y_train_cv.iloc[idx_train].copy()
    y_fold_valid = y_train_cv.iloc[idx_valid].copy()

    # ---------------------------------
    # OHE: fit SOLO con fold de train
    # ---------------------------------

    preprocessor_fold = ColumnTransformer(
        transformers=[
            (
                'cat',
                OneHotEncoder(handle_unknown='ignore'),
                categoricas
            ),
            (
                'num',
                'passthrough',
                numericas
            )
        ]
    )

    X_fold_train_encoded = preprocessor_fold.fit_transform(
        X_fold_train
    )

    X_fold_valid_encoded = preprocessor_fold.transform(
        X_fold_valid
    )

    # ---------------------------------
    # Convertir a DataFrame
    # ---------------------------------

    feature_names_fold = (
        preprocessor_fold.get_feature_names_out()
    )

    X_fold_train_encoded = pd.DataFrame(
        X_fold_train_encoded,
        columns=feature_names_fold
    )

    X_fold_valid_encoded = pd.DataFrame(
        X_fold_valid_encoded,
        columns=feature_names_fold
    )

    # ---------------------------------
    # Oversampling SOLO en fold train
    # ---------------------------------

    X_fold_up, y_fold_up = fp.upsample_array(
        X_fold_train_encoded,
        y_fold_train.to_numpy(),
        repeat=3
    )

    X_fold_up = pd.DataFrame(
        X_fold_up,
        columns=feature_names_fold
    )

    # ---------------------------------
    # Modelo
    # ---------------------------------

    modelo_cv = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=75,
        random_state=42,
        verbosity=-1
    )

    # ---------------------------------
    # Entrenamiento
    # ---------------------------------

    inicio = time.perf_counter()

    modelo_cv.fit(
        X_fold_up,
        y_fold_up
    )

    tiempo = time.perf_counter() - inicio

    # ---------------------------------
    # Probabilidades
    # ---------------------------------

    probas = modelo_cv.predict_proba(
        X_fold_valid_encoded
    )[:, 1]

    # ---------------------------------
    # ROC-AUC
    # ---------------------------------

    auc = roc_auc_score(
        y_fold_valid,
        probas
    )

    resultados_cv.append({
        'Fold': fold,
        'ROC-AUC': auc,
        'Tiempo entrenamiento (s)': tiempo
    })

# Crea  y muestra dataframe de resultados
resultados_cv = pd.DataFrame(resultados_cv)

display(resultados_cv)

# muestra resultados
print(
    f"ROC-AUC medio: "
    f"{resultados_cv['ROC-AUC'].mean():.6f}"
)

print(
    f"Desviación estándar: "
    f"{resultados_cv['ROC-AUC'].std():.6f}"
)

,Fold,ROC-AUC,Tiempo entrenamiento (s)
0,1,0.922171,3.524073
1,2,0.923144,2.010705
2,3,0.925368,3.546886
3,4,0.951454,1.667423
4,5,0.929531,1.227797


ROC-AUC medio: 0.930334
Desviación estándar: 0.012142


# Modelo final y test

**La hora de la verdad Test y ROC-AUC final**

evidencias para continuar:
* `'tenure_months'` aporta bastante.
* LightGBM fue claramente superior a los otros modelos.
* Oversampling fue la estrategia que arrojó el mejor ROC-AUC.
* Configuración que se decide llevar al final es:
    * `n_estimators=300`
    * `learning_rate=0.05`
    * `num_leaves=63`
    * `max_depth=-1`
    * `min_child_samples=75`
* CV: ROC-AUC medio = 0.930334 ± 0.012142
* TEST sigue intacto.

**Paso final: entrenar y evaluar en TEST**

Ahora se entrenará el modelo definitivo juntando TRAIN + VALID, manteniendo exactamente los hiperparámetros elegidos, y finalmente evaluar una sola vez sobre TEST.


In [45]:
# Crea conjuntos de entrenamiento finales
X_train_final = pd.concat(
    [X_train, X_valid],
    axis=0
).reset_index(drop=True)

y_train_final = pd.concat(
    [y_train, y_valid],
    axis=0
).reset_index(drop=True)

# comprueba cambios
print("X_train_final:", X_train_final.shape)
print("y_train_final:", y_train_final.shape)

# Se realiza el OHE para las columnas categóricas y aplioca fit solo en train
preprocessor_final = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categoricas
        ),
        (
            'num',
            'passthrough',
            numericas
        )
    ]
)

X_train_final_encoded = preprocessor_final.fit_transform(
    X_train_final
)

X_test_final_encoded = preprocessor_final.transform(
    X_test
)

# convertimos a df para evitar warnings del modelo
feature_names_final = (
    preprocessor_final.get_feature_names_out()
)

X_train_final_encoded = pd.DataFrame(
    X_train_final_encoded,
    columns=feature_names_final
)

X_test_final_encoded = pd.DataFrame(
    X_test_final_encoded,
    columns=feature_names_final
)

# comprueba cambios
print(X_train_final_encoded.shape)
print(X_test_final_encoded.shape)

# Oversampling solo en train
X_train_final_up, y_train_final_up = fp.upsample_array(
    X_train_final_encoded,
    y_train_final.to_numpy(),
    repeat=3
)

# Recupera nombres
X_train_final_up = pd.DataFrame(
    X_train_final_up,
    columns=feature_names_final
)

# Comprueba las clases
print(
    pd.Series(y_train_final_up).value_counts()
)


X_train_final: (6338, 18)
y_train_final: (6338,)
(6338, 26)
(705, 26)
1    5046
0    4656
Name: count, dtype: int64


In [46]:
# Entranamiento del modelo definitivo
## Definición del modelo final
modelo_final = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=75,
    random_state=42,
    verbosity=-1
)

## Entrenamiento del modelo final con tiempo
inicio = time.perf_counter()

modelo_final.fit(
    X_train_final_up,
    y_train_final_up
)

tiempo_final = time.perf_counter() - inicio

print(
    f"Tiempo de entrenamiento: {tiempo_final:.3f} segundos"
)


Tiempo de entrenamiento: 1.694 segundos


In [47]:
# Obtención de probabilidades
proba_test = modelo_final.predict_proba(
    X_test_final_encoded
)[:, 1]

# Cálculo y muestra de ROC-AUC
roc_auc_test = roc_auc_score(
    y_test,
    proba_test
)

print(
    f"ROC-AUC TEST: {roc_auc_test:.6f}"
)

# Obtención de todas las métricas
pred_test = (proba_test >= 0.5).astype(int)

f1_test = f1_score(
    y_test,
    pred_test,
    zero_division=0
)

precision_test = precision_score(
    y_test,
    pred_test,
    zero_division=0
)

recall_test = recall_score(
    y_test,
    pred_test,
    zero_division=0
)

accuracy_test = accuracy_score(
    y_test,
    pred_test
)

print(f"ROC-AUC : {roc_auc_test:.6f}")
print(f"F1      : {f1_test:.6f}")
print(f"Precision: {precision_test:.6f}")
print(f"Recall  : {recall_test:.6f}")
print(f"Accuracy: {accuracy_test:.6f}")

ROC-AUC TEST: 0.941063
ROC-AUC : 0.941063
F1      : 0.800000
Precision: 0.808743
Recall  : 0.791444
Accuracy: 0.895035


In [48]:
# Reporte de clasificación
print(
    classification_report(
        y_test,
        pred_test,
        target_names=['No churn', 'Churn'],
        zero_division=0
    )
)

              precision    recall  f1-score   support

    No churn       0.93      0.93      0.93       518
       Churn       0.81      0.79      0.80       187

    accuracy                           0.90       705
   macro avg       0.87      0.86      0.86       705
weighted avg       0.89      0.90      0.89       705



# Conclusiones finales

El proyecto permitió desarrollar un sistema de clasificación capaz de identificar clientes con riesgo de abandonar los servicios de Interconnect.

El análisis comenzó con la integración de cuatro fuentes de información: contratos, datos personales, servicios de Internet y servicios telefónicos. Después de realizar el EDA y el preprocesamiento, se identificó un desbalance entre las clases: aproximadamente el 26.5 % de los clientes habían cancelado el servicio.

Durante la ingeniería de características se creó `tenure_months`, una variable que representa la antigüedad aproximada del cliente. La comparación entre modelos con y sin esta característica mostró que su inclusión aportaba una mejora considerable en el desempeño predictivo.

Se compararon Logistic Regression, LinearSVC, Random Forest y LightGBM, utilizando diferentes estrategias para tratar el desbalance de clases. LightGBM presentó el mejor desempeño de forma consistente, especialmente al utilizar oversampling de la clase minoritaria.

La configuración final del modelo fue:

* `n_estimators = 300`
* `learning_rate = 0.05`
* `num_leaves = 63`
* `max_depth = -1`
* `min_child_samples = 75`
* `random_state = 42`

Antes de realizar la evaluación final se utilizó una validación cruzada estratificada de 5 folds. El modelo obtuvo:

**ROC-AUC medio = 0.930334 ± 0.012142**

Los resultados de los folds fueron:

| Fold |  ROC-AUC |
| ---: | -------: |
|    1 | 0.922171 |
|    2 | 0.923144 |
|    3 | 0.925368 |
|    4 | 0.951454 |
|    5 | 0.929531 |

La desviación estándar relativamente baja respecto al promedio indica que el desempeño se mantuvo estable entre las diferentes particiones utilizadas durante la validación cruzada.

Finalmente, se entrenó el modelo utilizando los conjuntos TRAIN y VALID y se realizó una única evaluación sobre TEST.

Los resultados finales fueron:

| Métrica     |         TEST |
| ----------- | -----------: |
| **ROC-AUC** | **0.941063** |
| **Recall**  | **0.791444** |
| F1          |     0.800000 |
| Precision   |     0.808743 |
| Accuracy    |     0.895035 |

El **ROC-AUC de 0.941063** supera ampliamente el requisito mínimo del proyecto de 0.75. Esto indica que el modelo tiene una elevada capacidad para distinguir entre clientes que permanecen y clientes que abandonan el servicio.

El **Recall de 0.791444** significa que el modelo identifica aproximadamente el 79.1 % de los clientes que efectivamente abandonan el servicio bajo el umbral de clasificación utilizado.

Desde una perspectiva de negocio, este modelo puede utilizarse como una herramienta de priorización para las campañas de retención. Interconnect podría utilizar las probabilidades generadas por el modelo para identificar clientes con mayor riesgo de churn y posteriormente ofrecer promociones, cambios de plan, beneficios o contacto personalizado.

Sin embargo, el modelo no debe interpretarse como una predicción determinista. Un Recall de 79.1 % también implica que existe un grupo de clientes que abandonará el servicio y que el modelo no identificará bajo el umbral utilizado. Por ello, las predicciones deberían utilizarse como apoyo a las decisiones del equipo de retención y combinarse con información comercial adicional.

## Conclusión general

El resultado final demuestra que el enfoque desarrollado es adecuado para el objetivo del proyecto. El modelo consiguió un **ROC-AUC de 0.941063 sobre datos de prueba no utilizados durante la selección del modelo**, acompañado de un Recall de 79.1 %.

Por lo tanto, Interconnect podría utilizar este modelo para crear una estrategia de retención basada en riesgo, concentrando recursos en los clientes con mayor probabilidad estimada de cancelación y utilizando posteriormente intervenciones comerciales para intentar reducir la fuga.

**Nota**
**Existen funciones para visualizar loe modelos en "funciones_personales.py"**